# 𝓣𝓱𝓮 IceTop-TNN assessment notebook.

In [ ]:
# 
# Adapted from IceTop-CNN's assessment notebook.
#

%matplotlib inline

import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# move out of nodebooks directory before importing icetoptnn
if (Path(os.getcwd()).name == 'notebooks'):
    os.chdir("..");

import icetoptnn.environment as tenv
import icetoptnn.util as tutil

# load environment.txt from project root dir
tenv.reload(Path('./environment.txt'))

# config

In [ ]:
# predicted outputs
conf_outputs = [ 'energy' ]

# idk see icetopcnn
conf_log_difference, conf_reco_percent = .1, 15

# range of logged energy values
conf_energy_range = (5, 8)

# setup

In [ ]:
# load model data
models = {}
for path in Path(tenv.get('ICETOP_TNN_MODEL_DIR')).iterdir():
    data = pd.read_csv(path / 'results.csv')
    
    outputs = []
    for output in iter(conf_outputs):
        if output in data:
            outputs.append(output)

    models[path.name] = {
        'data':    data,
        'outputs': outputs
    }

# data

In [ ]:
# calculate statistics for each output
for name in iter(models):
    data = models[name]

    print(f'Stats for {name}:')
    for output in data['outputs']:
        # logged predicted - true
        diff = np.log10(data['data'][output + '_pred']) - np.log10(data['data'][output])
        
        # predicted - true
        true_diff = data['data'][output + '_pred'] - data['data'][output]

        # percent diff in predicted - true
        percent_diff = 100 * true_diff / data['data'][output]

        # i don't know what these numbers mean, see icetop-cnn energy notebook :)
        std_dev_lo, median, std_dev_hi = np.percentile(diff, (16, 50, 84))

        # see icetopcnn
        diff_percentage = 100 * len(list(filter(lambda x: abs(x) <= conf_log_difference, diff))) / len(diff)
        percent_diff_percentage = 100 * len(list(filter(lambda x: abs(x) <= conf_reco_percent, percent_diff))) / len(percent_diff)
        
        # print info
        print(f'  Output "{output}":')
        print(f'    Resolution: {median:.3f} +{std_dev_hi:.3f} {std_dev_lo:.3f}')
        print(f'    Events reconstructed within a {conf_log_difference} in order of magnitude: {diff_percentage:.2f}%')
        print(f'    Events reconstructed within {conf_reco_percent}% of their true values: {percent_diff_percentage:.2f}%')

In [ ]:
# resolution graph
fig, axs = plt.subplots(len(conf_outputs))
if len(conf_outputs) == 1:
    axs = [axs]

# plot energy resolution of each output
for output, ax in iter(zip(conf_outputs, axs)):
    for name in iter(models):
        data = models[name]

        # add model to plot if it has the output
        if output in data['outputs']:
            ax.hist((np.log10(data['data'][output + '_pred']) - np.log10(data['data'][output])),
                    label=name,
                    histtype='step', linewidth=2,
                    log=True, density=True, range=(-2, 2), bins=101)
    
    # vertical line through x=0 (perfect reconstruction)
    ax.axvline()

    # add text
    ax.set_title(f'Resolution: {output}')
    ax.set_ylabel('Density');
    ax.set_xlabel('log10(reconstructed) - log10(truth)');

    # add legend
    ax.legend()

In [ ]:
# zoomed, unlogged resolution graph
fig, axs = plt.subplots(len(conf_outputs))
if len(conf_outputs) == 1:
    axs = [axs]

# plot energy resolution of each output
for output, ax in iter(zip(conf_outputs, axs)):
    for name in iter(models):
        data = models[name]

        # add model to plot if it has the output
        if output in data['outputs']:
            ax.hist((np.log10(data['data'][output + '_pred']) - np.log10(data['data'][output])),
                    label=name,
                    histtype='step', linewidth=2,
                    log=False, density=True, range=(-1, 1), bins=101)
    
    # vertical line through x=0 (perfect reconstruction)
    ax.axvline()

    # add text
    ax.set_title(f'Resolution: {output}')
    ax.set_ylabel('Density');
    ax.set_xlabel('log10(reconstructed) - log10(truth)');

    # add legend
    ax.legend()

In [ ]:
# histogram reco to truth for each output
for name in iter(models):
    data = models[name]

    for output in iter(data['outputs']):
        fig, ax = plt.subplots();

        # create histogram of reconstructed energy to true energy
        hist, _, _ = np.histogram2d(np.log10(data['data'][output + '_pred']), np.log10(data['data'][output]),
                                    bins=(200, 200))

        # remove empty cells
        hist = np.ma.masked_where(hist == 0, hist)

        # show histogram
        image = ax.imshow(hist, origin='lower', extent=(*conf_energy_range, *conf_energy_range))

        # line through x=y
        ax.plot(conf_energy_range, conf_energy_range, color='darkgrey')
        
        # add text
        ax.set_title(f'{output} in {name}')
        ax.set_ylabel('log10(reco)')
        ax.set_xlabel('log10(truth)')

        # colorbar
        fig.colorbar(image, ax=ax)